# Latent-group EM

Subjects come from `G` unknown groups, each with its own boundary template.
`BayesBreakMixtureClassifier` alternates E-step (exact responsibilities) and
M-step (responsibility-weighted max-sum DP), optimising the finite
template-mixture objective `ℓ_⋆` (`thm:em-monotone`).

In [ ]:
import numpy as np
from bayesbreak import BayesBreakGaussian, BayesBreakMixtureClassifier

rng = np.random.default_rng(0)
n = 60
# Group A: jump at index 30 (up).
group_a = [
    np.r_[rng.normal(-1.0, 0.2, 30), rng.normal(2.0, 0.2, 30)] for _ in range(8)
]
# Group B: jump at index 15 (down).
group_b = [
    np.r_[rng.normal(0.0, 0.2, 15), rng.normal(-2.0, 0.2, 45)] for _ in range(8)
]
sequences = group_a + group_b
labels_true = np.array([0]*8 + [1]*8)

## Fit the mixture with G=2 (correct G)

In [ ]:
mix = BayesBreakMixtureClassifier(
    BayesBreakGaussian(k_max=4),
    n_groups=2,
    max_iter=20,
    random_state=0,
).fit(sequences)

print('canonical permutation:', mix.canonical_permutation_)
print('pi:', mix.pi_)
for g, state in enumerate(mix.group_states_):
    print(f'  group {g}: k_g={state.k_g}, template={state.template}')

## Verify identifiability anchoring (`prop:latent-identifiability`)

The canonical anchor sorts groups by `k_g`, then lexicographically by
`t^{(g)}`. Two restarts converging to the same template multiset must
report them in identical order.

In [ ]:
mix_alt = BayesBreakMixtureClassifier(
    BayesBreakGaussian(k_max=4), n_groups=2, max_iter=20, random_state=42
).fit(sequences)

set_orig = {(s.k_g, tuple(s.template)) for s in mix.group_states_}
set_alt  = {(s.k_g, tuple(s.template)) for s in mix_alt.group_states_}
print('matched templates:', set_orig == set_alt)
if set_orig == set_alt:
    keys_orig = [(s.k_g, tuple(s.template)) for s in mix.group_states_]
    keys_alt  = [(s.k_g, tuple(s.template)) for s in mix_alt.group_states_]
    print('same canonical order:', keys_orig == keys_alt)

## Held-out G selection (`select_n_groups_by_holdout`)

Mitigates `rem:teicher-overspec` overspecification: at G > G*, two
distinct (π, τ) tuples can produce identical mixture densities.
Held-out marginal log-likelihood is the §5b recommended response.

In [ ]:
from bayesbreak import select_n_groups_by_holdout

sel = select_n_groups_by_holdout(
    BayesBreakGaussian(k_max=4), sequences,
    g_grid=(1, 2, 3),
    n_folds=4,
    random_state=0,
)
print('best_g =', sel.extra['best_g'])
for g, m, s in zip(sel.extra['g_grid'], sel.extra['mean_test_loglik'], sel.extra['std_test_loglik']):
    print(f'  G={g}: mean held-out log p(y) = {m:.2f}  (±{s:.2f})')